# 01 Pattern Generation

Merged notebook for modules 01a-01h in tutorial order.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 01a - Stick Patterns

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # User-facing controls for plot range
    MIN_ANGLE = 10
    MAX_ANGLE = 80

    # Initialize XRD calculator on NaCl
    pattern = XRDCalculator(wavelength="CuKa").get_pattern(
        Structure.from_file("data/cif/NaCl.cif"),
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Initialize plot
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot discrete peaks as vertical lines
    ax.vlines(pattern.x, 0, pattern.y, color="black", linewidth=6.5)

    # Formatting
    ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
    ax.set_ylim(0, max(pattern.y) * 1.05)
    ax.set_xlabel("2θ", fontsize=18, labelpad=12)
    ax.set_ylabel("Intensity", fontsize=18, labelpad=12)
    ax.tick_params(axis="both", labelsize=15)

    # Save plot
    plt.tight_layout()
    plt.savefig("NaCl_stick_pattern.png", dpi=200)
    print("\nSaved plot: NaCl_stick_pattern.png")


    """
    Try on your own:
    - Vary the wavelength (1.5406 Å is the CuKa value; you can specify others)
    - Load other structures and plot their XRD stick patterns
    - Check how peaks look at higher values of two-theta
    """


# 01a — Stick Patterns

This notebook introduces idealized XRD stick patterns using NaCl as a simple reference system.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
The vertical lines mark Bragg peak positions (2θ) and relative intensities for an ideal crystal.

In [ ]:
display(Image("NaCl_stick_pattern.png"))

## Summary
- Stick patterns show discrete reflection positions and intensities.
- Peak positions come from lattice geometry; heights come from structure factors.
- This is the baseline representation before instrumental/sample artifacts are added.

## Next Steps
Continue to the next section below in this notebook.

## 01b - Continuous Patterns

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Plotting range and peak width
    MIN_ANGLE = 10
    MAX_ANGLE = 80

    NUM_POINTS = 4000 # number of points in XRD pattern
    FWHM = 0.3 # full width at half maximum
    GAUSS_FRAC = 0.2 # fraction gaussian (vs. lorentzian)


    def gaussian(x, center, fwhm):
        sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        return np.exp(-0.5 * ((x - center) / sigma) ** 2)


    def lorentzian(x, center, fwhm):
        gamma = fwhm / 2.0
        return (gamma**2) / ((x - center) ** 2 + gamma**2)


    def pseudo_voigt(x, center, fwhm, eta):
        return (1.0 - eta) * gaussian(x, center, fwhm) + eta * lorentzian(x, center, fwhm)


    # Initialize XRD calculator on NaCl
    pattern = XRDCalculator(wavelength="CuKa").get_pattern(
        Structure.from_file("data/cif/NaCl.cif"),
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Extract discrete peak positions and intensities
    peak_pos = np.array(pattern.x)
    peak_intensity = np.array(pattern.y)

    # Build a high-resolution 2theta grid for a continuous profile
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)
    continuous_intensity = np.zeros_like(two_theta_grid)

    # Broaden each stick peak into a pseudo-Voigt line shape
    for t, i in zip(peak_pos, peak_intensity):
        continuous_intensity += i * pseudo_voigt(two_theta_grid, t, FWHM, GAUSS_FRAC)

    # Keep peak scale similar to the original stick pattern.
    continuous_intensity *= peak_intensity.max() / continuous_intensity.max()

    # Initialize plot
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot continuous profile as a filled curve with an outline
    ax.fill_between(two_theta_grid, 0, continuous_intensity, color="blue", alpha=0.25)
    ax.plot(two_theta_grid, continuous_intensity, color="darkblue", linewidth=2.2)

    # Formatting
    ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
    ax.set_ylim(0, continuous_intensity.max() * 1.05)
    ax.set_xlabel("2θ", fontsize=18, labelpad=12)
    ax.set_ylabel("Intensity", fontsize=18, labelpad=12)
    ax.tick_params(axis="both", labelsize=15)

    # Save plot
    plt.tight_layout()
    plt.savefig("NaCl_continuous_pattern.png", dpi=200)
    print("\nSaved plot: NaCl_continuous_pattern.png")

    """
    Try on your own:
    - Use broader peaks (larger FWHM)
    - Changing the Gaussian/Lorentzian fraction (GAUSS_FRAC)
    - Loading other structures and plotting their continuous XRD patterns
    """


# 01b — Continuous Patterns

We broaden stick peaks into a pseudo-Voigt profile to mimic realistic measured line shapes.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
The spectrum becomes continuous, and peak overlap begins to matter for phase identification.

In [ ]:
display(Image("NaCl_continuous_pattern.png"))

## Summary
- Broadening turns ideal sticks into realistic peak envelopes.
- FWHM and Gaussian/Lorentzian mixing control profile shape.
- Continuous profiles are used by most fitting and ML pipelines.

## Next Steps
Continue to the next section below in this notebook.

## 01c - Peak Splitting

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Plotting range and peak width
    MIN_ANGLE = 10
    MAX_ANGLE = 80

    NUM_POINTS = 4000  # number of points in XRD pattern
    FWHM = 0.1  # full width at half maximum
    GAUSS_FRAC = 0.2  # fraction gaussian (vs. lorentzian)

    # Cu Kalpha doublet values used in galaxi
    CU_KA1_WAVELENGTH = 1.5405929
    CU_KA2_WAVELENGTH = 1.5444260
    CU_KA1_KA2_RATIO = 2.0  # Kalpha1:Kalpha2 intensity ratio


    def gaussian(x, center, fwhm):
        sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        return np.exp(-0.5 * ((x - center) / sigma) ** 2)


    def lorentzian(x, center, fwhm):
        gamma = fwhm / 2.0
        return (gamma**2) / ((x - center) ** 2 + gamma**2)


    def pseudo_voigt(x, center, fwhm, eta):
        return (1.0 - eta) * gaussian(x, center, fwhm) + eta * lorentzian(x, center, fwhm)


    # Load NaCl structure once
    structure = Structure.from_file("data/cif/NaCl.cif")

    # Compute stick patterns separately for Cu Kalpha1 and Kalpha2
    pattern_ka1 = XRDCalculator(wavelength=CU_KA1_WAVELENGTH).get_pattern(
        structure,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )
    pattern_ka2 = XRDCalculator(wavelength=CU_KA2_WAVELENGTH).get_pattern(
        structure,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Extract discrete peak positions and intensities
    peak_pos_ka1 = np.array(pattern_ka1.x)
    peak_intensity_ka1 = np.array(pattern_ka1.y)

    peak_pos_ka2 = np.array(pattern_ka2.x)
    peak_intensity_ka2 = np.array(pattern_ka2.y)

    # Kalpha1/Kalpha2 intensity weights
    weight_ka1 = CU_KA1_KA2_RATIO / (1.0 + CU_KA1_KA2_RATIO)
    weight_ka2 = 1.0 / (1.0 + CU_KA1_KA2_RATIO)

    # Build a high-resolution 2theta grid for a continuous profile
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)
    continuous_intensity = np.zeros_like(two_theta_grid)

    # Broaden Kalpha1 peaks into pseudo-Voigt line shapes
    for t, i in zip(peak_pos_ka1, peak_intensity_ka1):
        continuous_intensity += weight_ka1 * i * pseudo_voigt(two_theta_grid, t, FWHM, GAUSS_FRAC)

    # Broaden Kalpha2 peaks into pseudo-Voigt line shapes
    for t, i in zip(peak_pos_ka2, peak_intensity_ka2):
        continuous_intensity += weight_ka2 * i * pseudo_voigt(two_theta_grid, t, FWHM, GAUSS_FRAC)

    # Keep a similar intensity scale as previous examples.
    continuous_intensity *= 100.0 / continuous_intensity.max()

    # Initialize plot
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot continuous profile as a filled curve with an outline
    ax.fill_between(two_theta_grid, 0, continuous_intensity, color="blue", alpha=0.25)
    ax.plot(two_theta_grid, continuous_intensity, color="darkblue", linewidth=1.5)

    # Formatting
    ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
    ax.set_ylim(0, continuous_intensity.max() * 1.05)
    ax.set_xlabel("2θ", fontsize=18, labelpad=12)
    ax.set_ylabel("Intensity", fontsize=18, labelpad=12)
    ax.tick_params(axis="both", labelsize=15)

    # Save plot
    plt.tight_layout()
    plt.savefig("NaCl_peak_splitting_pattern.png", dpi=200)
    print("\nSaved plot: NaCl_peak_splitting_pattern.png")

    """
    Let's zoom in on a peak at high two-theta
    to better visualize the splitting effect
    """
    # Initialize plot
    fig, ax = plt.subplots(figsize=(6, 4))

    # Plot continuous profile as a filled curve with an outline
    ax.fill_between(two_theta_grid, 0, continuous_intensity, color="blue", alpha=0.25)
    ax.plot(two_theta_grid, continuous_intensity, color="darkblue", linewidth=1.5)

    # Zoomed in
    ax.set_xlim(75, 77.5)
    ax.set_ylim(0, 28)

    # Save plot
    plt.tight_layout()
    plt.savefig("NaCl_zoomed.png", dpi=200)
    print("\nSaved plot: NaCl_zoomed.png")

    """
    Try on your own:
    - Include higher two-theta to see more visible splitting
    - Check how the peak width (larger or smaller FWHM) affects splitting
    """


# 01c — Peak Splitting

This example adds Cu Kα1/Kα2 doublet splitting and highlights the effect at higher angles.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
The zoomed view shows subtle doublet separation that increases in visibility at larger 2θ.

In [ ]:
display(Image("NaCl_peak_splitting_pattern.png"))
display(Image("NaCl_zoomed.png"))

## Summary
- Kα splitting creates near-duplicate peak components.
- High-angle reflections make splitting easiest to see.
- Ignoring splitting can bias line-profile comparisons.

## Next Steps
Continue to the next section below in this notebook.

## 01d - Peak Broadening

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Input structure and plotting range
    CIF_FILE = "data/cif/TiO2.cif"
    MIN_ANGLE = 10
    MAX_ANGLE = 80
    NUM_POINTS = 4000

    # Scherrer broadening settings
    WAVELENGTH_ANGSTROM = 1.5406  # Cu Kalpha
    K_FACTOR = 0.9
    PROFILES = [
        ("Large particles", 30.0),
        ("Moderate size", 10.0),
        ("Very small", 5.0),
    ]

    # Color map used across the 3 particle-size profiles
    PROFILE_CMAP = LinearSegmentedColormap.from_list(
        "blue_purple_red",
        ["#1f4ed8", "#7e22ce", "#dc2626"],
    )


    def scherrer_fwhm_deg(two_theta_deg, crystallite_size_nm):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
        beta_rad = K_FACTOR * wavelength_nm / (crystallite_size_nm * np.cos(theta_rad))
        return np.rad2deg(beta_rad)


    def gaussian_unit_area(two_theta_grid, centers, fwhm_deg):
        sigma = fwhm_deg / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        dx = two_theta_grid[:, None] - centers[None, :]
        return np.exp(-0.5 * (dx / sigma[None, :]) ** 2) / (
            sigma[None, :] * np.sqrt(2.0 * np.pi)
        )


    def broaden_pattern(two_theta_grid, peak_pos, peak_intensity, size_nm):
        fwhm = scherrer_fwhm_deg(peak_pos, size_nm)
        profile = gaussian_unit_area(two_theta_grid, peak_pos, fwhm)
        intensity = profile @ peak_intensity
        return 100.0 * intensity / intensity.max()


    # Initialize XRD calculator on TiO2
    pattern = XRDCalculator(wavelength=WAVELENGTH_ANGSTROM).get_pattern(
        Structure.from_file(CIF_FILE),
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Extract discrete peak positions and intensities
    peak_pos = np.array(pattern.x)
    peak_intensity = np.array(pattern.y)
    peak_intensity = 100.0 * peak_intensity / peak_intensity.max()

    # Build a high-resolution 2theta grid for continuous profiles
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    # Initialize a 3-panel plot (one panel per crystallite size)
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 7), sharex=True)
    colors = [PROFILE_CMAP(v) for v in np.linspace(0.0, 1.0, len(PROFILES))]

    # Broaden the same stick pattern using different crystallite sizes
    for ax, color, (_, size_nm) in zip(axes, colors, PROFILES):
        continuous_intensity = broaden_pattern(two_theta_grid, peak_pos, peak_intensity, size_nm)
        ax.fill_between(two_theta_grid, 0, continuous_intensity, color=color, alpha=0.25)
        ax.plot(two_theta_grid, continuous_intensity, color=color, linewidth=2.2)
        ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=18, labelpad=12)
        ax.tick_params(axis="both", labelsize=15)

    axes[-1].set_xlabel("2θ", fontsize=18, labelpad=12)

    # Save plot
    output = "TiO2_peak_broadening.png"
    plt.tight_layout()
    plt.savefig(output, dpi=200)
    print(f"\nLoaded CIF: {CIF_FILE}")
    print(f"Saved plot: {output}")
    print("Smaller crystallite size gives broader peaks (larger Scherrer FWHM).")

    """
    Try on your own:
    - Change the particle sizes in PROFILES and re-plot the XRD patterns
    """


# 01d — Peak Broadening

Here we use a Scherrer-style model to show how crystallite size impacts line width.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
Smaller crystallite size yields visibly broader peaks and reduced apparent resolution.

In [ ]:
display(Image("TiO2_peak_broadening.png"))

## Summary
- Finite domain size broadens diffraction peaks.
- Broadening can obscure nearby reflections.
- Crystallite-size effects are a key source of mismatch in real data.

## Next Steps
Continue to the next section below in this notebook.

## 01e - Baseline Effects

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Reproducible random noise
    np.random.seed(42)

    # Input structure and plotting range
    CIF_FILE = "data/cif/LiMnO2.cif"
    MIN_ANGLE = 10
    MAX_ANGLE = 80
    NUM_POINTS = 4000

    # Peak profile settings
    ETA = 0.5  # pseudo-Voigt mixing (0 = Gaussian, 1 = Lorentzian)
    CRYSTALLITE_SIZE_NM = 20.0
    MICROSTRAIN = 0.001

    # Instrument broadening (Caglioti; matches galaxi defaults)
    U = 0.04
    V = -0.01
    W = 0.006

    # Cu Kalpha split settings (galaxi defaults)
    CU_KA1_WAVELENGTH = 1.5405929
    CU_KA2_WAVELENGTH = 1.5444260
    KA1_KA2_RATIO = 2.0  # Kalpha1:Kalpha2 intensity ratio

    # Background and noise settings
    CHEB_COEFFS = np.array([1.0, -0.3, 0.15, -0.05, 0.02])
    BACKGROUND_SCALE = 0.24
    NOISE_LEVEL = 0.005
    BASELINE_OFFSET = 0.01


    def instrumental_fwhm(two_theta_deg):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        tan_theta = np.tan(theta_rad)
        fwhm_sq = U * tan_theta**2 + V * tan_theta + W
        return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


    def size_fwhm(two_theta_deg):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        wavelength_nm = CU_KA1_WAVELENGTH / 10.0
        beta_rad = 0.9 * wavelength_nm / (CRYSTALLITE_SIZE_NM * np.cos(theta_rad))
        return np.rad2deg(beta_rad)


    def strain_fwhm(two_theta_deg):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        beta_rad = 4.0 * MICROSTRAIN * np.tan(theta_rad)
        return np.rad2deg(beta_rad)


    def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
        dx = two_theta_grid[:, None] - centers[None, :]
        sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
        gamma = np.clip(fwhm / 2.0, 1e-6, None)
        gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
        lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
        return (1.0 - eta) * gauss + eta * lorentz


    # Load structure once
    structure = Structure.from_file(CIF_FILE)

    # Compute stick patterns separately for Cu Kalpha1 and Kalpha2
    pattern_ka1 = XRDCalculator(wavelength=CU_KA1_WAVELENGTH).get_pattern(
        structure,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )
    pattern_ka2 = XRDCalculator(wavelength=CU_KA2_WAVELENGTH).get_pattern(
        structure,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Extract discrete peak positions and intensities
    peak_pos_ka1 = np.array(pattern_ka1.x)
    peak_intensity_ka1 = np.array(pattern_ka1.y)
    peak_pos_ka2 = np.array(pattern_ka2.x)
    peak_intensity_ka2 = np.array(pattern_ka2.y)

    # Build a high-resolution 2theta grid for a continuous profile
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    # Combine instrument + size + strain broadening for each peak
    fwhm_ka1 = np.sqrt(
        instrumental_fwhm(peak_pos_ka1) ** 2
        + size_fwhm(peak_pos_ka1) ** 2
        + strain_fwhm(peak_pos_ka1) ** 2
    )
    fwhm_ka2 = np.sqrt(
        instrumental_fwhm(peak_pos_ka2) ** 2
        + size_fwhm(peak_pos_ka2) ** 2
        + strain_fwhm(peak_pos_ka2) ** 2
    )

    # Kalpha1/Kalpha2 intensity weights
    weight_ka1 = KA1_KA2_RATIO / (1.0 + KA1_KA2_RATIO)
    weight_ka2 = 1.0 / (1.0 + KA1_KA2_RATIO)

    # Broaden and combine Kalpha1 + Kalpha2 peaks
    profile_ka1 = pseudo_voigt_profile(two_theta_grid, peak_pos_ka1, fwhm_ka1, ETA) @ peak_intensity_ka1
    profile_ka2 = pseudo_voigt_profile(two_theta_grid, peak_pos_ka2, fwhm_ka2, ETA) @ peak_intensity_ka2
    peaks = weight_ka1 * profile_ka1 + weight_ka2 * profile_ka2
    peaks *= max(peak_intensity_ka1.max(), peak_intensity_ka2.max()) / peaks.max()

    # Add smooth Chebyshev background
    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    background_shape = np.polynomial.chebyshev.chebval(x_cheb, CHEB_COEFFS)
    background_shape -= background_shape.min()
    background_shape /= background_shape.max()
    background = BACKGROUND_SCALE * peaks.max() * background_shape

    # Add Gaussian counting noise
    noise = np.random.randn(NUM_POINTS) * NOISE_LEVEL * peaks.max()

    # Final intensity (keep positive baseline for plotting)
    intensity = peaks + background + noise
    intensity -= intensity.min()
    intensity += BASELINE_OFFSET * peaks.max()

    # Initialize plot
    fig, ax = plt.subplots(figsize=(8, 4))

    # Plot continuous profile as a filled curve with an outline
    ax.fill_between(two_theta_grid, 0, intensity, color="black", alpha=0.15)
    ax.plot(two_theta_grid, intensity, color="black", linewidth=2.2)

    # Formatting
    ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
    ax.set_ylim(0, intensity.max() * 1.05)
    ax.set_xlabel("2θ", fontsize=18, labelpad=12)
    ax.set_ylabel("Intensity", fontsize=18, labelpad=12)
    ax.tick_params(axis="both", labelsize=15)

    # Save plot
    plt.tight_layout()
    plt.savefig("LiMnO2_xrd_with_background.png", dpi=200)
    print("\nSaved plot: LiMnO2_xrd_with_background.png")

    """
    Try on your own:
    - Change CHEB_COEFFS to make the baseline flatter or more curved
    - Increase/decrease BACKGROUND_SCALE to control baseline height
    - Increase/decrease NOISE_LEVEL to simulate cleaner/noisier data
    """


# 01e — Baseline Effects

This notebook adds background curvature and noise on top of broadened peaks.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
Baseline shape and counting noise can dominate weak regions of the pattern.

In [ ]:
display(Image("LiMnO2_xrd_with_background.png"))

## Summary
- Background modeling is essential before quantitative comparison.
- Noise level influences peak detection robustness.
- Even simple baseline terms can noticeably alter correlations.

## Next Steps
Continue to the next section below in this notebook.

## 01f - Lattice Strain

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Input structure and plotting range
    CIF_FILE = "data/cif/TiO2.cif"
    MIN_ANGLE = 10
    MAX_ANGLE = 80
    NUM_POINTS = 4000

    # Peak broadening settings
    WAVELENGTH_ANGSTROM = 1.5406  # Cu Kalpha
    K_FACTOR = 0.9
    CRYSTALLITE_SIZE_NM = 17.5

    # Tetragonal-preserving strain
    # Keep a = b so symmetry is not broken.
    STRAIN_LEVEL = -0.015
    OUT_OF_PLANE_STRAIN = -0.010

    PATTERNS = [
        ("TiO2", "#1f4ed8"),
        ("TiO2 strained", "#dc2626"),
    ]


    def scherrer_fwhm_deg(two_theta_deg, crystallite_size_nm):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        wavelength_nm = WAVELENGTH_ANGSTROM / 10.0
        beta_rad = K_FACTOR * wavelength_nm / (crystallite_size_nm * np.cos(theta_rad))
        return np.rad2deg(beta_rad)


    def gaussian_unit_area(two_theta_grid, centers, fwhm_deg):
        sigma = fwhm_deg / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        dx = two_theta_grid[:, None] - centers[None, :]
        return np.exp(-0.5 * (dx / sigma[None, :]) ** 2) / (
            sigma[None, :] * np.sqrt(2.0 * np.pi)
        )


    def broaden_pattern(two_theta_grid, peak_pos, peak_intensity, size_nm):
        fwhm = scherrer_fwhm_deg(peak_pos, size_nm)
        profile = gaussian_unit_area(two_theta_grid, peak_pos, fwhm)
        intensity = profile @ peak_intensity
        return 100.0 * intensity / intensity.max()


    # Load TiO2 once
    tio2 = Structure.from_file(CIF_FILE)

    # Build a strained copy directly from TiO2 (no second CIF needed)
    tio2_strained = tio2.copy()
    tio2_strained.apply_strain([STRAIN_LEVEL, STRAIN_LEVEL, OUT_OF_PLANE_STRAIN])

    structures = [tio2, tio2_strained]

    # Initialize XRD calculator and 2theta grid
    calc = XRDCalculator(wavelength=WAVELENGTH_ANGSTROM)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)

    # Initialize plot
    fig, ax = plt.subplots(figsize=(8, 2.5))

    # Compute and plot broadened profiles for original + strained TiO2
    for structure, (label, color) in zip(structures, PATTERNS):
        pattern = calc.get_pattern(structure, two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        peak_pos = np.array(pattern.x)
        peak_intensity = np.array(pattern.y)
        peak_intensity = 100.0 * peak_intensity / peak_intensity.max()

        continuous_intensity = broaden_pattern(
            two_theta_grid, peak_pos, peak_intensity, CRYSTALLITE_SIZE_NM
        )
        ax.fill_between(two_theta_grid, 0, continuous_intensity, color=color, alpha=0.25)
        ax.plot(two_theta_grid, continuous_intensity, color=color, linewidth=2.2, label=label)

    # Formatting (kept same look as original figure)
    ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
    ax.set_ylim(0, 105)
    ax.tick_params(axis="both", labelsize=15)
    ax.set_yticks([])

    # Save plot
    output = "TiO2_strained-xrd.png"
    plt.tight_layout()
    plt.savefig(output, dpi=200)
    print(f"\nLoaded CIF: {CIF_FILE}")
    print(f"Saved plot: {output}")


    """
    Try on your own:
    - Change STRAIN_LEVEL and OUT_OF_PLANE_STRAIN and re-plot
    - Keep the in-plane strain the same for a and b so tetragonal symmetry is preserved
    """


# 01f — Lattice Strain

We compare unstrained and strained TiO2 to visualize lattice-parameter-driven peak shifts.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
Strain changes d-spacings, shifting peak positions while preserving overall phase identity.

In [ ]:
display(Image("TiO2_strained-xrd.png"))

## Summary
- Lattice strain primarily shifts peak positions.
- Shift direction and magnitude depend on strain tensor components.
- Position-sensitive methods must handle small systematic offsets.

## Next Steps
Continue to the next section below in this notebook.

## 01g - Texture

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Input structure and plotting range
    CIF_FILE = "data/cif/TiO2.cif"
    MIN_ANGLE = 10
    MAX_ANGLE = 80
    NUM_POINTS = 4000

    # Peak profile settings
    FWHM = 0.4  # full width at half maximum
    GAUSS_FRAC = 0.2  # fraction gaussian (vs. lorentzian)

    # Texture settings (March-Dollase)
    PREFERRED_ORIENTATION = (0, 0, 1)
    MARCH_DOLLASE_R = 0.65  # 1.0 = random orientation (no texture)


    def gaussian(x, center, fwhm):
        sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        return np.exp(-0.5 * ((x - center) / sigma) ** 2)


    def lorentzian(x, center, fwhm):
        gamma = fwhm / 2.0
        return (gamma**2) / ((x - center) ** 2 + gamma**2)


    def pseudo_voigt(x, center, fwhm, eta):
        return (1.0 - eta) * gaussian(x, center, fwhm) + eta * lorentzian(x, center, fwhm)


    def march_dollase_factor(hkls, preferred_orientation, r):
        # Same core form used in galaxi:
        # P(alpha) = (r^2 cos^2(alpha) + sin^2(alpha)/r)^(-3/2)
        hkls = np.asarray(hkls, dtype=float)
        preferred = np.asarray(preferred_orientation, dtype=float)

        preferred_mag = np.linalg.norm(preferred)
        if preferred_mag < 1e-12:
            raise ValueError("Preferred orientation vector must be non-zero.")

        hkl_mag = np.linalg.norm(hkls, axis=1)
        hkl_mag = np.clip(hkl_mag, 1e-12, None)

        cos_alpha = (hkls @ preferred) / (hkl_mag * preferred_mag)
        cos_alpha = np.clip(cos_alpha, -1.0, 1.0)
        sin_alpha_sq = 1.0 - cos_alpha**2

        r = max(float(r), 1e-6)
        numerator = r * r * cos_alpha**2 + sin_alpha_sq / r
        numerator = np.clip(numerator, 1e-12, None)
        return numerator ** (-1.5)


    def continuous_profile(two_theta_grid, peak_pos, peak_intensity, fwhm, eta):
        intensity = np.zeros_like(two_theta_grid)
        for t, i in zip(peak_pos, peak_intensity):
            intensity += i * pseudo_voigt(two_theta_grid, t, fwhm, eta)
        return 100.0 * intensity / intensity.max()


    # Load TiO2 structure and compute stick pattern
    pattern = XRDCalculator(wavelength="CuKa").get_pattern(
        Structure.from_file(CIF_FILE),
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    # Extract discrete peak positions, intensities, and hkls
    peak_pos = np.array(pattern.x)
    peak_intensity = np.array(pattern.y)
    peak_hkls = np.array([h[0]["hkl"] for h in pattern.hkls], dtype=float)

    # No-texture intensities (random orientation)
    intensity_no_texture = peak_intensity.copy()

    # Apply preferred orientation correction to stick intensities
    texture_scale = march_dollase_factor(peak_hkls, PREFERRED_ORIENTATION, MARCH_DOLLASE_R)
    intensity_with_texture = peak_intensity * texture_scale

    # Build a high-resolution 2theta grid for continuous profiles
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)
    continuous_no_texture = continuous_profile(
        two_theta_grid, peak_pos, intensity_no_texture, FWHM, GAUSS_FRAC
    )
    continuous_with_texture = continuous_profile(
        two_theta_grid, peak_pos, intensity_with_texture, FWHM, GAUSS_FRAC
    )

    # Initialize a 2-panel plot: without texture (top), with texture (bottom)
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True)

    axes[0].fill_between(two_theta_grid, 0, continuous_no_texture, color="#1f4ed8", alpha=0.25)
    axes[0].plot(two_theta_grid, continuous_no_texture, color="#1f4ed8", linewidth=2.2)
    axes[0].set_title("Without texture", fontsize=16, pad=4)

    axes[1].fill_between(two_theta_grid, 0, continuous_with_texture, color="#dc2626", alpha=0.25)
    axes[1].plot(two_theta_grid, continuous_with_texture, color="#dc2626", linewidth=2.2)
    axes[1].set_title(
        f"With Texture",
        fontsize=16,
        pad=4,
    )

    # Formatting
    for ax in axes:
        ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=16, labelpad=10)
        ax.tick_params(axis="both", labelsize=13)

    axes[-1].set_xlabel("2θ", fontsize=16, labelpad=10)

    # Save plot
    output = "TiO2_texture_comparison.png"
    plt.tight_layout()
    plt.savefig(output, dpi=200)
    print(f"\nLoaded CIF: {CIF_FILE}")
    print(f"Preferred orientation: {PREFERRED_ORIENTATION}")
    print(f"March-Dollase r (textured panel): {MARCH_DOLLASE_R}")
    print(f"Saved plot: {output}")


    """
    Try on your own:
    - Change PREFERRED_ORIENTATION, e.g., (1, 0, 0), (1, 1, 0), (1, 1, 1)
    - Apply weaker/stronger texture by changing MARCH_DOLLASE_R (closer/farther from 1.0)
    """


# 01g — Texture

Preferred orientation is modeled with a March-Dollase correction to modify reflection intensities.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
Peak intensities change selectively, even when peak positions remain nearly unchanged.

In [ ]:
display(Image("TiO2_texture_comparison.png"))

## Summary
- Texture reweights peaks based on reflection orientation.
- Intensity-only matching is sensitive to preferred orientation.
- Texture-aware augmentation improves model robustness.

## Next Steps
Continue to the next section below in this notebook.

## 01h - All Artifacts

In [ ]:
from IPython.display import Image, display

# Inline tutorial script
# Run run_demo() in the execution cell below.
def run_demo():
    # For handling arrays
    import numpy as np

    # For plotting
    import matplotlib.pyplot as plt

    # To load structures and compute XRD patterns
    from pymatgen.core import Structure
    from pymatgen.analysis.diffraction.xrd import XRDCalculator

    # Input structure and plotting range
    CIF_FILE = "data/cif/Li2CO3.cif"
    OUTPUT_FILE = "Li2CO3_ideal_vs_artifacts.png"
    MIN_ANGLE = 10
    MAX_ANGLE = 80
    NUM_POINTS = 4000

    # Cu Kalpha split settings
    CU_KA1_WAVELENGTH = 1.5405929
    CU_KA2_WAVELENGTH = 1.5444260
    KA1_KA2_RATIO = 2.0

    # Idealized profile settings
    IDEAL_FWHM = 0.30
    IDEAL_ETA = 0.2

    # Mixed-artifact settings
    ETA = 0.55
    CRYSTALLITE_SIZE_NM = 18.0
    MICROSTRAIN = 0.0015
    U, V, W = 0.018, -0.004, 0.004
    UNIFORM_SHIFT_RANGE = (-0.04, 0.04)
    SAMPLE_DISPLACEMENT_RANGE_MM = (-0.15, 0.15)
    GONIOMETER_RADIUS_MM = 240.0
    PREFERRED_ORIENTATION = (0, 0, 1)
    MARCH_DOLLASE_R = 0.7
    STRAIN_IN_PLANE = -0.010
    STRAIN_OUT_OF_PLANE = -0.006
    BACKGROUND_SCALE = 0.16
    DIFFUSE_SCALE = 0.10
    AMORPHOUS_SCALE = 0.12
    NOISE_LEVEL = 0.012
    BASELINE_OFFSET = 0.01

    # Chebyshev background shape
    CHEB_COEFFS = np.array([1.0, -0.3, 0.15, -0.05, 0.02], dtype=float)

    # Reproducible artifact sampling
    np.random.seed(42)


    def gaussian(x, center, fwhm):
        sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
        return np.exp(-0.5 * ((x - center) / sigma) ** 2)


    def lorentzian(x, center, fwhm):
        gamma = fwhm / 2.0
        return (gamma**2) / ((x - center) ** 2 + gamma**2)


    def pseudo_voigt(x, center, fwhm, eta):
        return (1.0 - eta) * gaussian(x, center, fwhm) + eta * lorentzian(x, center, fwhm)


    def pseudo_voigt_profile(two_theta_grid, centers, fwhm, eta):
        dx = two_theta_grid[:, None] - centers[None, :]
        sigma = np.clip(fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0))), 1e-6, None)
        gamma = np.clip(fwhm / 2.0, 1e-6, None)
        gauss = np.exp(-0.5 * (dx / sigma[None, :]) ** 2)
        lorentz = (gamma[None, :] ** 2) / (dx**2 + gamma[None, :] ** 2)
        return (1.0 - eta) * gauss + eta * lorentz


    def instrumental_fwhm(two_theta_deg):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        tan_theta = np.tan(theta_rad)
        fwhm_sq = U * tan_theta**2 + V * tan_theta + W
        return np.sqrt(np.clip(fwhm_sq, 1e-4, None))


    def size_fwhm(two_theta_deg, size_nm):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        wavelength_nm = CU_KA1_WAVELENGTH / 10.0
        beta_rad = 0.9 * wavelength_nm / (size_nm * np.cos(theta_rad))
        return np.rad2deg(beta_rad)


    def strain_fwhm(two_theta_deg, microstrain):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        beta_rad = 4.0 * microstrain * np.tan(theta_rad)
        return np.rad2deg(beta_rad)


    def sample_displacement_shift(two_theta_deg, displacement_mm):
        theta_rad = np.deg2rad(two_theta_deg / 2.0)
        d_relative_change = displacement_mm / GONIOMETER_RADIUS_MM * np.cos(theta_rad) ** 2
        return np.rad2deg(-d_relative_change * np.tan(theta_rad))


    def march_dollase_factor(hkls, preferred_orientation, r):
        hkls = np.asarray(hkls, dtype=float)
        preferred = np.asarray(preferred_orientation, dtype=float)
        hkl_mag = np.clip(np.linalg.norm(hkls, axis=1), 1e-12, None)
        pref_mag = np.clip(np.linalg.norm(preferred), 1e-12, None)
        cos_alpha = np.clip((hkls @ preferred) / (hkl_mag * pref_mag), -1.0, 1.0)
        sin_alpha_sq = 1.0 - cos_alpha**2
        r = max(float(r), 1e-6)
        return np.clip(r * r * cos_alpha**2 + sin_alpha_sq / r, 1e-12, None) ** (-1.5)


    def normalize_0_100(y):
        y = np.asarray(y).flatten()
        return 100.0 * y / np.clip(y.max(), 1e-12, None)


    # Load Li2CO3 and make a strained copy for the artifact panel
    structure = Structure.from_file(CIF_FILE)
    structure_strained = structure.copy()
    structure_strained.apply_strain([STRAIN_IN_PLANE, STRAIN_IN_PLANE, STRAIN_OUT_OF_PLANE])

    # Build idealized continuous profile from a CuKa stick pattern
    ideal_pattern = XRDCalculator(wavelength="CuKa").get_pattern(
        structure,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )
    ideal_peak_pos = np.array(ideal_pattern.x)
    ideal_peak_intensity = np.array(ideal_pattern.y)
    two_theta_grid = np.linspace(MIN_ANGLE, MAX_ANGLE, NUM_POINTS)
    ideal_intensity = np.zeros_like(two_theta_grid)
    for t, i in zip(ideal_peak_pos, ideal_peak_intensity):
        ideal_intensity += i * pseudo_voigt(two_theta_grid, t, IDEAL_FWHM, IDEAL_ETA)
    ideal_intensity *= ideal_peak_intensity.max() / np.clip(ideal_intensity.max(), 1e-12, None)
    ideal_intensity = normalize_0_100(ideal_intensity)

    # Build artifact-rich profile: splitting + strain + texture + broadening
    pattern_ka1 = XRDCalculator(wavelength=CU_KA1_WAVELENGTH).get_pattern(
        structure_strained,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )
    pattern_ka2 = XRDCalculator(wavelength=CU_KA2_WAVELENGTH).get_pattern(
        structure_strained,
        two_theta_range=(MIN_ANGLE, MAX_ANGLE),
    )

    pos_ka1 = np.array(pattern_ka1.x)
    int_ka1 = np.array(pattern_ka1.y)
    hkls_ka1 = np.array([h[0]["hkl"] for h in pattern_ka1.hkls], dtype=float)

    pos_ka2 = np.array(pattern_ka2.x)
    int_ka2 = np.array(pattern_ka2.y)
    hkls_ka2 = np.array([h[0]["hkl"] for h in pattern_ka2.hkls], dtype=float)

    int_ka1 *= march_dollase_factor(hkls_ka1, PREFERRED_ORIENTATION, MARCH_DOLLASE_R)
    int_ka2 *= march_dollase_factor(hkls_ka2, PREFERRED_ORIENTATION, MARCH_DOLLASE_R)

    uniform_shift = np.random.uniform(*UNIFORM_SHIFT_RANGE)
    displacement = np.random.uniform(*SAMPLE_DISPLACEMENT_RANGE_MM)
    pos_ka1 = pos_ka1 + uniform_shift + sample_displacement_shift(pos_ka1, displacement)
    pos_ka2 = pos_ka2 + uniform_shift + sample_displacement_shift(pos_ka2, displacement)

    fwhm_ka1 = np.sqrt(
        instrumental_fwhm(pos_ka1) ** 2
        + size_fwhm(pos_ka1, CRYSTALLITE_SIZE_NM) ** 2
        + strain_fwhm(pos_ka1, MICROSTRAIN) ** 2
    )
    fwhm_ka2 = np.sqrt(
        instrumental_fwhm(pos_ka2) ** 2
        + size_fwhm(pos_ka2, CRYSTALLITE_SIZE_NM) ** 2
        + strain_fwhm(pos_ka2, MICROSTRAIN) ** 2
    )

    profile_ka1 = pseudo_voigt_profile(two_theta_grid, pos_ka1, fwhm_ka1, ETA) @ int_ka1
    profile_ka2 = pseudo_voigt_profile(two_theta_grid, pos_ka2, fwhm_ka2, ETA) @ int_ka2

    weight_ka1 = KA1_KA2_RATIO / (1.0 + KA1_KA2_RATIO)
    weight_ka2 = 1.0 / (1.0 + KA1_KA2_RATIO)
    artifact_peaks = weight_ka1 * profile_ka1 + weight_ka2 * profile_ka2
    artifact_peaks *= 100.0 / np.clip(artifact_peaks.max(), 1e-12, None)

    # Add smooth background + diffuse + amorphous hump + noise
    x_cheb = 2.0 * (two_theta_grid - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0
    background = np.polynomial.chebyshev.chebval(x_cheb, CHEB_COEFFS)
    background = background - background.min()
    background = background / np.clip(background.max(), 1e-12, None)
    background = BACKGROUND_SCALE * artifact_peaks.max() * background

    theta = np.deg2rad(two_theta_grid / 2.0)
    diffuse = DIFFUSE_SCALE * artifact_peaks.max() * np.exp(-2.0 * np.sin(theta) ** 2)
    amorphous_hump = AMORPHOUS_SCALE * artifact_peaks.max() * np.exp(
        -0.5 * ((two_theta_grid - 25.0) / 7.5) ** 2
    )
    noise = np.random.randn(NUM_POINTS) * NOISE_LEVEL * artifact_peaks.max()

    artifact_intensity = artifact_peaks + background + diffuse + amorphous_hump + noise
    artifact_intensity = artifact_intensity - artifact_intensity.min()
    artifact_intensity = artifact_intensity + BASELINE_OFFSET * artifact_peaks.max()
    artifact_intensity = normalize_0_100(artifact_intensity)

    # Initialize a 2-panel plot: idealized (top), mixed artifacts (bottom)
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True)

    axes[0].fill_between(two_theta_grid, 0, ideal_intensity, color="#1f4ed8", alpha=0.25)
    axes[0].plot(two_theta_grid, ideal_intensity, color="#1f4ed8", linewidth=2.2)
    axes[0].set_title("Idealized", fontsize=16, pad=4)

    axes[1].fill_between(two_theta_grid, 0, artifact_intensity, color="#dc2626", alpha=0.25)
    axes[1].plot(two_theta_grid, artifact_intensity, color="#dc2626", linewidth=2.2)
    axes[1].set_title("With all artifacts mixed in", fontsize=16, pad=4)

    # Formatting
    for ax in axes:
        ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=16, labelpad=10)
        ax.tick_params(axis="both", labelsize=13)

    axes[-1].set_xlabel("2θ", fontsize=16, labelpad=10)

    # Save plot
    plt.tight_layout()
    plt.savefig(OUTPUT_FILE, dpi=200)
    print(f"\nLoaded CIF: {CIF_FILE.split('/')[-1]}")
    print(f"Saved plot: {OUTPUT_FILE}")


    """
    Try on your own:
    - Change PREFERRED_ORIENTATION and MARCH_DOLLASE_R to explore texture strength
    - Increase/decrease NOISE_LEVEL, BACKGROUND_SCALE, and AMORPHOUS_SCALE
    - Change STRAIN_IN_PLANE / STRAIN_OUT_OF_PLANE and compare peak-position shifts
    """


# 01h — All Artifacts Together

This combined example mixes splitting, broadening, background, noise, strain, and texture in one profile.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
run_demo()


## What To Observe
The artifact-rich profile looks much closer to an experimental pattern than the idealized baseline.

In [ ]:
display(Image("Li2CO3_ideal_vs_artifacts.png"))

## Summary
- Realistic patterns are a superposition of multiple artifacts.
- Single-artifact assumptions often underfit real data variability.
- This motivates synthetic augmentation for ML and DL modules.

## Next Steps
Continue to **02 — Conventional Methods**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/02_Conventional-Methods.ipynb)